# Stage 6: SHAP Explainability**Explainable Skin Lesion Classification Using SHAP and LIME**\Responsible AI Course ProjectSHAP (SHapley Additive exPlanations) is grounded in cooperative game theory.Each pixel gets a **Shapley value** — its marginal contribution to the prediction.Unlike LIME's coarse superpixels, SHAP produces **pixel-level continuous heatmaps**showing fine-grained attribution across the entire image.

## 6.1 Import Libraries

In [ ]:
import osimport jsonimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.colors as mcolorsimport seaborn as snsimport tensorflow as tffrom tensorflow.keras.models import load_modelimport shapimport warningswarnings.filterwarnings('ignore')# Force TF to not use all GPU memorygpus = tf.config.list_physical_devices('GPU')if gpus:    for gpu in gpus:        tf.config.experimental.set_memory_growth(gpu, True)print(f"TensorFlow: {tf.__version__}")print(f"SHAP: {shap.__version__}")print("Libraries loaded!")

## 6.2 Load Model and Data

In [ ]:
# Load trained modelmodel = load_model('models/mobilenetv2_skin_lesion_final.h5')print(f"Model loaded: {model.name}")# Load preprocessed datadata = np.load("outputs/stage2_preprocessed_data.npz")X_train = data['X_train']X_test = data['X_test']y_test = data['y_test']# Load configwith open("outputs/stage2_config.json", 'r') as f:    config = json.load(f)idx_to_label = {int(k): v for k, v in config['idx_to_label'].items()}label_to_idx = config['label_to_idx']CLASS_NAMES = config['class_names']NUM_CLASSES = config['num_classes']CLASSES_SORTED = [idx_to_label[i] for i in range(NUM_CLASSES)]# Load predictions from Stage 4preds = np.load("outputs/stage4_predictions.npz")y_pred = preds['y_pred']y_pred_proba = preds['y_pred_proba']y_true = preds['y_true']correct_mask = preds['correct_mask']# Load LIME results to use the SAME imageswith open('outputs/lime_explanations/lime_results.json', 'r') as f:    lime_results = json.load(f)os.makedirs("outputs/shap_explanations", exist_ok=True)print(f"Test set: {X_test.shape[0]} images")print(f"Train set (for background): {X_train.shape[0]} images")

## 6.3 Prepare Background Dataset for SHAPSHAP's GradientExplainer needs a **background dataset** — a representative sample of training images. It uses these as the baseline to compute how each pixel in the test image shifts the prediction away from the "average" prediction.We use 100 randomly sampled training images. More = better accuracy but slower.

In [ ]:
# Sample 100 background images from training setnp.random.seed(42)bg_indices = np.random.choice(len(X_train), size=100, replace=False)background = X_train[bg_indices]print(f"Background dataset: {background.shape}")print(f"Background pixel range: [{background.min():.3f}, {background.max():.3f}]")

## 6.4 Create SHAP GradientExplainerGradientExplainer computes expected gradients — an extension of integrated gradients that uses a distribution of baselines (our background set) instead of a single baseline.This gives us smooth, pixel-level attribution maps.

In [ ]:
# Create the GradientExplainer# This wraps our model and uses the background set as referenceexplainer = shap.GradientExplainer(model, background)print("SHAP GradientExplainer created!")print(f"  Model: {model.name}")print(f"  Background samples: {len(background)}")print(f"  Output classes: {NUM_CLASSES}")

## 6.5 Select the SAME Images as LIMEWe use the exact same image indices from Stage 5 so we can do a fair side-by-side comparison in Stage 7.

In [ ]:
# Reconstruct the same image indices used in LIMEcorrect_indices_by_class = lime_results['correct_indices']wrong_indices_list = lime_results['wrong_indices']print("Using same images as LIME Stage 5:")print(f"Correct predictions:")for cls, indices in correct_indices_by_class.items():    print(f"  {cls:6s}: indices {indices}")print(f"Misclassifications: {wrong_indices_list}")# Collect all indicesall_correct_indices = []for cls, indices in correct_indices_by_class.items():    all_correct_indices.extend(indices)total_images = len(all_correct_indices) + len(wrong_indices_list)print(f"Total images to explain: {total_images}")

## 6.6 Generate SHAP Values for Correct Predictions**Expected time: ~60-120 seconds per image.**SHAP returns an array of shape  — a full attribution map per class per image.

In [ ]:
print("=" * 60)print("GENERATING SHAP VALUES: CORRECT PREDICTIONS")print("=" * 60)print("(~60-120 sec per image)")correct_shap_values = {}count = 0total = len(all_correct_indices)for cls_label, indices in correct_indices_by_class.items():    correct_shap_values[cls_label] = []    for idx in indices:        count += 1        print(f"  [{count}/{total}] Class: {cls_label}, Index: {idx}...", end=" ", flush=True)        # SHAP expects shape (n, 224, 224, 3)        test_image = X_test[idx:idx+1]        # Compute SHAP values        sv = explainer.shap_values(test_image)        # sv is a list of NUM_CLASSES arrays, each of shape (1, 224, 224, 3)        # Or sometimes a single array of shape (1, 224, 224, 3, NUM_CLASSES)        correct_shap_values[cls_label].append((idx, sv))        print("Done!")print(f"Completed {count} correct prediction SHAP explanations!")

## 6.7 Generate SHAP Values for Misclassifications

In [ ]:
print("" + "=" * 60)print("GENERATING SHAP VALUES: MISCLASSIFICATIONS")print("=" * 60)print("(~60-120 sec per image)")wrong_shap_values = []for i, idx in enumerate(wrong_indices_list):    print(f"  [{i+1}/{len(wrong_indices_list)}] Index: {idx}, True: {idx_to_label[y_true[idx]]}, Pred: {idx_to_label[y_pred[idx]]}...", end=" ", flush=True)    test_image = X_test[idx:idx+1]    sv = explainer.shap_values(test_image)    wrong_shap_values.append((idx, sv))    print("Done!")print(f"Completed {len(wrong_shap_values)} misclassification SHAP explanations!")

## 6.8 Helper Function: Extract SHAP HeatmapSHAP returns attribution per channel (R, G, B). We sum across channels to get a single importance map per class, then extract the map for the predicted class.

In [ ]:
def get_shap_heatmap(shap_values, image_index, class_index):    """    Extract a 2D SHAP heatmap for a specific class from SHAP output.    SHAP output format varies by version:      - List of arrays: shap_values[class_idx] has shape (n, 224, 224, 3)      - Single array: shape (n, 224, 224, 3, n_classes)    Returns: 2D heatmap of shape (224, 224)    """    if isinstance(shap_values, list):        # List format: one array per class        sv = shap_values[class_index][image_index]  # (224, 224, 3)    else:        # Single array format        if shap_values.ndim == 5:            sv = shap_values[image_index, :, :, :, class_index]  # (224, 224, 3)        else:            sv = shap_values[image_index]  # (224, 224, 3)    # Sum across RGB channels to get single importance map    heatmap = sv.sum(axis=-1)  # (224, 224)    return heatmap# Test the helpertest_idx, test_sv = list(correct_shap_values.values())[0][0]test_heatmap = get_shap_heatmap(test_sv, 0, y_pred[test_idx])print(f"Heatmap shape: {test_heatmap.shape}")print(f"Heatmap range: [{test_heatmap.min():.6f}, {test_heatmap.max():.6f}]")print("Helper function working!")

## 6.9 Visualize SHAP: Correct Predictions (All Classes)For each image we show:- **Original image**- **SHAP heatmap**: red = supports prediction, blue = opposes- **SHAP overlay**: heatmap superimposed on the original image- **Absolute SHAP**: intensity of attribution regardless of direction

In [ ]:
def visualize_shap_explanation(image, shap_values, true_label, pred_label, pred_idx, confidence, index, save_path=None):    """    Create a 4-panel SHAP visualization for a single image.    """    heatmap = get_shap_heatmap(shap_values, 0, pred_idx)    # Symmetric color scale centered at 0    abs_max = max(abs(heatmap.min()), abs(heatmap.max()))    if abs_max == 0:        abs_max = 1e-10    fig, axes = plt.subplots(1, 4, figsize=(20, 5))    # Panel 1: Original    axes[0].imshow(image)    axes[0].set_title(f"OriginalTrue: {true_label} | Pred: {pred_label}Conf: {confidence:.3f}", fontsize=10)    axes[0].axis('off')    # Panel 2: SHAP heatmap (red-blue diverging)    im1 = axes[1].imshow(heatmap, cmap='bwr', vmin=-abs_max, vmax=abs_max)    axes[1].set_title("SHAP heatmap(red=supports, blue=opposes)", fontsize=10)    axes[1].axis('off')    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)    # Panel 3: SHAP overlay on original    axes[2].imshow(image, alpha=0.6)    im2 = axes[2].imshow(heatmap, cmap='bwr', alpha=0.5, vmin=-abs_max, vmax=abs_max)    axes[2].set_title("SHAP overlay(attribution on image)", fontsize=10)    axes[2].axis('off')    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)    # Panel 4: Absolute SHAP (importance magnitude)    abs_heatmap = np.abs(heatmap)    im3 = axes[3].imshow(abs_heatmap, cmap='hot')    axes[3].set_title("Absolute SHAP(importance magnitude)", fontsize=10)    axes[3].axis('off')    plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)    plt.tight_layout()    if save_path:        plt.savefig(save_path, dpi=150, bbox_inches='tight')    plt.show()# Visualize all correct predictionsprint("=" * 60)print("SHAP EXPLANATIONS: CORRECT PREDICTIONS")print("=" * 60)for cls_label, sv_list in correct_shap_values.items():    for idx, sv in sv_list:        true_label = idx_to_label[y_true[idx]]        pred_label = idx_to_label[y_pred[idx]]        pred_idx = y_pred[idx]        conf = np.max(y_pred_proba[idx])        save_path = f"outputs/shap_explanations/shap_correct_{cls_label}_idx{idx}.png"        print(f"--- {cls_label} ({CLASS_NAMES[cls_label]}) | Confidence: {conf:.3f} ---")        visualize_shap_explanation(X_test[idx], sv, true_label, pred_label, pred_idx, conf, idx, save_path)print("All correct prediction SHAP visualizations saved!")

## 6.10 Visualize SHAP: MisclassificationsFor misclassified images, we show SHAP for the **predicted** class (what the model thought) — revealing which pixels led the model astray.

In [ ]:
print("=" * 60)print("SHAP EXPLANATIONS: MISCLASSIFICATIONS")print("=" * 60)for idx, sv in wrong_shap_values:    true_label = idx_to_label[y_true[idx]]    pred_label = idx_to_label[y_pred[idx]]    pred_idx = y_pred[idx]    conf = np.max(y_pred_proba[idx])    save_path = f"outputs/shap_explanations/shap_wrong_true{true_label}_pred{pred_label}_idx{idx}.png"    print(f"--- TRUE: {true_label} -> PREDICTED: {pred_label} | Confidence: {conf:.3f} ---")    visualize_shap_explanation(X_test[idx], sv, true_label, pred_label, pred_idx, conf, idx, save_path)print("All misclassification SHAP visualizations saved!")

## 6.11 SHAP: Class Comparison GridAll 7 classes side by side — the SHAP equivalent of the LIME grid from Stage 5.

In [ ]:
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(20, NUM_CLASSES * 4))col_titles = ["Original", "SHAP heatmap", "SHAP overlay", "Absolute SHAP"]for row_idx, cls_label in enumerate(CLASSES_SORTED):    if cls_label in correct_shap_values and len(correct_shap_values[cls_label]) > 0:        idx, sv = correct_shap_values[cls_label][0]        image = X_test[idx]        pred_idx = y_pred[idx]        conf = np.max(y_pred_proba[idx])        heatmap = get_shap_heatmap(sv, 0, pred_idx)        abs_max = max(abs(heatmap.min()), abs(heatmap.max()))        if abs_max == 0:            abs_max = 1e-10        # Original        axes[row_idx][0].imshow(image)        axes[row_idx][0].set_title(f"{cls_label}({CLASS_NAMES[cls_label][:18]})Conf: {conf:.3f}", fontsize=9)        axes[row_idx][0].axis('off')        # SHAP heatmap        axes[row_idx][1].imshow(heatmap, cmap='bwr', vmin=-abs_max, vmax=abs_max)        if row_idx == 0:            axes[row_idx][1].set_title("SHAP heatmap", fontsize=9)        axes[row_idx][1].axis('off')        # Overlay        axes[row_idx][2].imshow(image, alpha=0.6)        axes[row_idx][2].imshow(heatmap, cmap='bwr', alpha=0.5, vmin=-abs_max, vmax=abs_max)        if row_idx == 0:            axes[row_idx][2].set_title("SHAP overlay", fontsize=9)        axes[row_idx][2].axis('off')        # Absolute        axes[row_idx][3].imshow(np.abs(heatmap), cmap='hot')        if row_idx == 0:            axes[row_idx][3].set_title("Absolute SHAP", fontsize=9)        axes[row_idx][3].axis('off')    else:        for col in range(4):            axes[row_idx][col].text(0.5, 0.5, f"No data{cls_label}", ha='center', va='center')            axes[row_idx][col].axis('off')plt.suptitle("SHAP Explanations Across All 7 Classes", fontsize=14, fontweight='bold', y=1.01)plt.tight_layout()plt.savefig('outputs/shap_explanations/shap_all_classes_grid.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: outputs/shap_explanations/shap_all_classes_grid.png")

## 6.12 SHAP: Multi-Class AttributionFor a single image, show SHAP attributions for ALL 7 classes — this reveals what the model "sees" for each possible diagnosis, not just the predicted one.

In [ ]:
# Pick one interesting imagedemo_idx = all_correct_indices[0]demo_sv_cls = list(correct_shap_values.keys())[0]demo_idx, demo_sv = correct_shap_values[demo_sv_cls][0]demo_image = X_test[demo_idx]fig, axes = plt.subplots(1, NUM_CLASSES + 1, figsize=(28, 4))# Originalaxes[0].imshow(demo_image)axes[0].set_title(f"OriginalTrue: {idx_to_label[y_true[demo_idx]]}Pred: {idx_to_label[y_pred[demo_idx]]}", fontsize=9)axes[0].axis('off')# SHAP for each classfor cls_idx in range(NUM_CLASSES):    heatmap = get_shap_heatmap(demo_sv, 0, cls_idx)    abs_max = max(abs(heatmap.min()), abs(heatmap.max()))    if abs_max == 0:        abs_max = 1e-10    axes[cls_idx + 1].imshow(demo_image, alpha=0.5)    axes[cls_idx + 1].imshow(heatmap, cmap='bwr', alpha=0.6, vmin=-abs_max, vmax=abs_max)    prob = y_pred_proba[demo_idx][cls_idx]    is_pred = " *" if cls_idx == y_pred[demo_idx] else ""    axes[cls_idx + 1].set_title(f"{idx_to_label[cls_idx]}{is_pred}p={prob:.3f}", fontsize=9)    axes[cls_idx + 1].axis('off')plt.suptitle("SHAP Attribution for All 7 Classes on One Image (* = predicted class)",             fontsize=12, fontweight='bold')plt.tight_layout()plt.savefig('outputs/shap_explanations/shap_multiclass_attribution.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: outputs/shap_explanations/shap_multiclass_attribution.png")

## 6.13 SHAP: Top Pixel Regions AnalysisIdentify the percentage of the image that SHAP considers most important by thresholding the absolute SHAP values.

In [ ]:
print("=" * 60)print("SHAP: TOP PIXEL REGION ANALYSIS")print("=" * 60)focus_percentages = []for cls_label, sv_list in correct_shap_values.items():    for idx, sv in sv_list:        pred_idx = y_pred[idx]        heatmap = get_shap_heatmap(sv, 0, pred_idx)        abs_heatmap = np.abs(heatmap)        # What percentage of pixels hold 80% of total attribution?        total_attr = abs_heatmap.sum()        if total_attr == 0:            continue        flat = abs_heatmap.flatten()        sorted_flat = np.sort(flat)[::-1]        cumsum = np.cumsum(sorted_flat)        threshold_idx = np.searchsorted(cumsum, 0.8 * total_attr)        pct_pixels = (threshold_idx + 1) / len(flat) * 100        focus_percentages.append(pct_pixels)        print(f"  {cls_label:6s} idx={idx}: top {pct_pixels:.1f}% of pixels hold 80% of attribution")if focus_percentages:    print(f"  Average: {np.mean(focus_percentages):.1f}% of pixels hold 80% of total SHAP attribution")    print(f"  This means the model focuses on a {'small' if np.mean(focus_percentages) < 30 else 'broad'} region of the image")

## 6.14 SHAP: Stability ComparisonUnlike LIME, SHAP with GradientExplainer is **deterministic** given the same background set. Let's verify this by running SHAP twice on the same image.

In [ ]:
# Run SHAP twice on the same imagestability_idx = all_correct_indices[0]stability_image = X_test[stability_idx:stability_idx+1]print(f"Stability test on image index {stability_idx}")print("Running SHAP twice...")sv_run1 = explainer.shap_values(stability_image)sv_run2 = explainer.shap_values(stability_image)hm1 = get_shap_heatmap(sv_run1, 0, y_pred[stability_idx])hm2 = get_shap_heatmap(sv_run2, 0, y_pred[stability_idx])# Comparediff = np.abs(hm1 - hm2)correlation = np.corrcoef(hm1.flatten(), hm2.flatten())[0, 1]print(f"  Max absolute difference: {diff.max():.8f}")print(f"  Mean absolute difference: {diff.mean():.8f}")print(f"  Pearson correlation: {correlation:.6f}")if correlation > 0.99:    print(f"  SHAP is highly stable (correlation > 0.99)")    print(f"  This is a key advantage over LIME for reproducibility.")else:    print(f"  Some variation detected (expected with gradient sampling)")# Visualizefig, axes = plt.subplots(1, 4, figsize=(20, 4))abs_max = max(abs(hm1.min()), abs(hm1.max()))axes[0].imshow(X_test[stability_idx])axes[0].set_title("Original", fontsize=10)axes[0].axis('off')axes[1].imshow(hm1, cmap='bwr', vmin=-abs_max, vmax=abs_max)axes[1].set_title("SHAP Run 1", fontsize=10)axes[1].axis('off')axes[2].imshow(hm2, cmap='bwr', vmin=-abs_max, vmax=abs_max)axes[2].set_title("SHAP Run 2", fontsize=10)axes[2].axis('off')axes[3].imshow(diff, cmap='Reds')axes[3].set_title(f"Absolute DifferenceCorr: {correlation:.4f}", fontsize=10)axes[3].axis('off')plt.suptitle("SHAP Stability Test: Two runs on same image", fontsize=12, fontweight='bold')plt.tight_layout()plt.savefig('outputs/shap_explanations/shap_stability_test.png', dpi=150, bbox_inches='tight')plt.show()print("Saved: outputs/shap_explanations/shap_stability_test.png")

## 6.15 Save SHAP Results for Stage 7

In [ ]:
# Save SHAP heatmaps for comparison in Stage 7shap_heatmaps = {}for cls_label, sv_list in correct_shap_values.items():    shap_heatmaps[cls_label] = []    for idx, sv in sv_list:        pred_idx = y_pred[idx]        heatmap = get_shap_heatmap(sv, 0, pred_idx)        shap_heatmaps[cls_label].append((idx, heatmap))wrong_shap_heatmaps = []for idx, sv in wrong_shap_values:    pred_idx = y_pred[idx]    heatmap = get_shap_heatmap(sv, 0, pred_idx)    wrong_shap_heatmaps.append((idx, heatmap))# Save as numpyall_heatmaps_correct = {}for cls, data_list in shap_heatmaps.items():    for idx, hm in data_list:        all_heatmaps_correct[str(idx)] = hmall_heatmaps_wrong = {}for idx, hm in wrong_shap_heatmaps:    all_heatmaps_wrong[str(idx)] = hmnp.savez_compressed('outputs/shap_explanations/shap_heatmaps_correct.npz', **all_heatmaps_correct)np.savez_compressed('outputs/shap_explanations/shap_heatmaps_wrong.npz', **all_heatmaps_wrong)# Save summaryshap_results = {    'correct_indices': {cls: [idx for idx, _ in sv_list] for cls, sv_list in correct_shap_values.items()},    'wrong_indices': [idx for idx, _ in wrong_shap_values],    'focus_pct_avg': float(np.mean(focus_percentages)) if focus_percentages else 0,    'stability_correlation': float(correlation),}with open('outputs/shap_explanations/shap_results.json', 'w') as f:    json.dump(shap_results, f, indent=2)print("Saved: outputs/shap_explanations/shap_heatmaps_correct.npz")print("Saved: outputs/shap_explanations/shap_heatmaps_wrong.npz")print("Saved: outputs/shap_explanations/shap_results.json")

## 6.16 Stage 6 Summary

In [ ]:
shap_files = os.listdir('outputs/shap_explanations')n_correct = sum(len(v) for v in correct_shap_values.values())n_wrong = len(wrong_shap_values)print("=" * 60)print("STAGE 6 SUMMARY")print("=" * 60)print(f"""SHAP Explainability Complete!Explanations Generated:  Correct predictions: {n_correct} images  Misclassifications:  {n_wrong} images  Total:               {n_correct + n_wrong} SHAP explanationsSHAP Configuration:  Method: GradientExplainer  Background samples: 100 (from training set)  Output: pixel-level attribution heatmaps (224x224)Visualizations Generated:  - Individual 4-panel explanations (heatmap, overlay, absolute)  - All-classes comparison grid  - Multi-class attribution (all 7 classes for one image)  - Stability test (2 runs, correlation: {correlation:.4f})Key Findings:  - SHAP focuses on ~{np.mean(focus_percentages):.0f}% of pixels for 80% of attribution  - Stability correlation: {correlation:.4f} (much more stable than LIME)  - Pixel-level detail reveals fine-grained patternsFiles saved: {len(shap_files)} files in outputs/shap_explanations/Next: Stage 7 - LIME vs SHAP Comparative Analysis""")